In [1]:
import pandas as pd
from datetime import datetime

In [22]:
#### THIS IS THE ONLY CELL CONTAINING ELEMENTS THAT NEED TO BE CHANGED
    # 1) the directory and Excel sheet number containing underway IFCB log data
    # 2) the directory and Excel sheet number containing discrete IFCB log data
    # 3) the cruise name/number (format example: AR82)
    # 4) the ifcb instrument number used for OOI discrete and underway sampling (format example: AR82)
        # this notebook is written assuming one instrument is used for both IFCB sampling sampling types
    # The directory containing either 5) seperate or 6) merged bottle files from the cruise

# Read in and standardize Underway IFCB data
processed_underway_ifcb_data = pd.read_excel('IFCB_Logs/Pioneer-22_AR98_IFCB_Log_2026-03-19_Ver_1-00.xlsx', sheet_name=0, header=3)   # change me (1)
processed_underway_ifcb_data = processed_underway_ifcb_data.fillna('')
processed_underway_ifcb_data.columns = processed_underway_ifcb_data.columns.str.strip()


# Read in and standardize Discrete IFCB data
processed_discrete_ifcb_data = pd.read_excel('IFCB_Logs/Pioneer-22_AR98_IFCB_Log_2026-03-19_Ver_1-00.xlsx', sheet_name=1, header=3)  # change me (2)
processed_discrete_ifcb_data = processed_discrete_ifcb_data.fillna('')
processed_discrete_ifcb_data.columns = processed_discrete_ifcb_data.columns.str.strip()

cruise = 'AR98'  # change me (3)
ifcb_instrument_num = 'IFCB199'  # change me (4)

# Read in the bottle files (the datetimes from here are needed for discrete samples)
    # (It's easier to complete steps related to bottle file processesing after the cruise/after ship data recovery)
# Define the folder with your .bl files
bottle_folder_path = 'Bottle_Data/'   # change me (5)
# Or the merged bottle file 
merged_bottle_file = pd.read_csv('Bottle_Data/Pioneer-22_AR98A_CTD_Bottle_Data_2025-12-18_ACR.csv')    # change me (6)


In [23]:
processed_discrete_ifcb_data.columns

Index(['Filename', 'HDR Comment', 'Sample Type', 'Cruise Leg', 'Site', 'Cast',
       'Niskin', 'IFCB Bottle', 'Target Cast Depth', 'Trip Depth',
       '# Triggers', '# ROIs', 'Run Time', 'Inhibit Time', 'Sample Time',
       'Volume Analyzed', 'ROIs/ml', 'Cast Start Latitude',
       'Cast Start Longitude', 'Notes'],
      dtype='object')

In [24]:
# Ensure Cast and Bottle numbers in the Discrete data df are interpreted as ints and preview the data

processed_discrete_ifcb_data['Cast'] = pd.to_numeric(processed_discrete_ifcb_data['Cast'], errors='coerce')
processed_discrete_ifcb_data['Niskin'] = pd.to_numeric(processed_discrete_ifcb_data['Niskin'], errors='coerce')
processed_discrete_ifcb_data = processed_discrete_ifcb_data.dropna(subset=['Cast', 'Niskin'])
processed_discrete_ifcb_data['Cast'] = processed_discrete_ifcb_data['Cast'].astype(int)
processed_discrete_ifcb_data['Niskin'] = processed_discrete_ifcb_data['Niskin'].astype(int)


processed_discrete_ifcb_data

,Filename,HDR Comment,Sample Type,Cruise Leg,Site,Cast,Niskin,IFCB Bottle,Target Cast Depth,Trip Depth,# Triggers,# ROIs,Run Time,Inhibit Time,Sample Time,Volume Analyzed,ROIs/ml,Cast Start Latitude,Cast Start Longitude,Notes
0,D20251101T164030_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,1,6,2,7,7.3,3698,3381,1200.866944,306.844688,894.022257,3.725093,907.628410,35.949567,-75.126083,LTER IFCB
1,D20251101T170419_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,1,6,2,7,7.3,3631,3309,1200.964583,300.425556,900.539028,3.752246,881.871830,35.949567,-75.126083,LTER IFCB
2,D20251101T172810_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,1,6,2,7,7.3,3705,3396,1201.778889,306.580000,895.198889,3.729995,910.456894,35.949567,-75.126083,LTER IFCB
3,D20251101T180159_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,1,2,11,bottom,27.0,3419,3144,1200.889583,283.876562,917.013021,3.820888,822.845459,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample
4,D20251101T183013_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,1,2,11,bottom,27.0,3422,3090,1200.994167,284.636094,916.358073,3.818159,809.290628,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,D20251116T014257_IFCB199,ar98b_c4n5_b2_cn_7m,discrete_water_sample,b,CP12CNSW,4,5,2,7,7.4,2496,2479,1200.884444,207.981337,992.903108,4.137096,599.212547,35.949933,-75.128200,
82,D20251116T020646_IFCB199,ar98b_c4n5_b2_cn_7m,discrete_water_sample,b,CP12CNSW,4,5,2,7,7.4,2434,2408,1200.947917,202.684010,998.263906,4.159433,578.925068,35.949933,-75.128200,
83,D20251116T023347_IFCB199,ar98b_c4n1_b1_cn_bottom,discrete_water_sample,b,CP12CNSW,4,1,1,bottom,27.3,1998,1972,1200.907917,166.444097,1034.463819,4.310266,457.512376,35.949933,-75.128200,
84,D20251116T025737_IFCB199,ar98b_c4n1_b1_cn_bottom,discrete_water_sample,b,CP12CNSW,4,1,1,bottom,27.3,1977,1933,1200.916111,164.649861,1036.266250,4.317776,447.684174,35.949933,-75.128200,


In [25]:
# Preview the Underway data

processed_underway_ifcb_data

,Filename,HDR Comment,Site,Intake Pump,# Triggers,# ROIs,Run Time,Inhibit Time,Sample Time,Volume Analyzed,ROIs/ml,Ship Latitude,Ship Longitude,Notes
0,D20251030T170909_IFCB199,ar98a_underway,,R/V Armstrong diaphragm pump,5284,4713,1201.528611,439.009687,762.518924,3.177162,1483.399251,41.210,-71.233,
1,D20251030T173245_IFCB199,ar98a_underway,,R/V Armstrong diaphragm pump,4154,3664,1201.593611,344.458160,857.135451,3.571398,1025.928864,41.158,-71.298,
2,D20251030T175621_IFCB199,ar98a_underway,,R/V Armstrong diaphragm pump,3673,3248,1201.048889,305.949236,895.099653,3.729582,870.875100,41.104,-71.367,
3,D20251030T181956_IFCB199,ar98a_underway,,R/V Armstrong diaphragm pump,3214,2846,1201.006250,267.862569,933.143681,3.888099,731.977309,41.050,-71.436,
4,D20251030T184331_IFCB199,ar98a_underway,,R/V Armstrong diaphragm pump,2916,2622,1200.907222,243.616736,957.290486,3.988710,657.355326,40.999,-71.502,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984,D20251119T091627_IFCB199,ar98b_underway,,R/V Armstrong diaphragm pump,6994,6989,1201.314306,582.004375,619.309931,2.580458,2708.433883,41.114,-70.893,
985,D20251119T094017_IFCB199,ar98b_underway,,R/V Armstrong diaphragm pump,6489,6363,1201.755000,540.033715,661.721285,2.757172,2307.799424,41.175,-70.883,
986,D20251119T100408_IFCB199,ar98b_underway,,R/V Armstrong diaphragm pump,6356,6221,1200.895278,530.149514,670.745764,2.794774,2225.940260,41.230,-70.886,
987,D20251119T102758_IFCB199,ar98b_underway,,R/V Armstrong diaphragm pump,6426,6337,1200.885139,535.086285,665.798854,2.774162,2284.293508,41.274,-70.889,


In [7]:
# Get datetimes from bottle data files

"""# Get list of all .bl files
bl_files = [os.path.join(bottle_folder_path, f) for f in os.listdir(bottle_folder_path) if f.lower().endswith('.bl')]

# Initialize list to store all rows
all_bottle_data = []

for filepath in bl_files:
    filename = os.path.basename(filepath)
    
    # Extract cruise and cast from the filename
    try:
        cruise_portion = filename.split('_')[0]      # e.g., 'ar87a'
        cast_portion = filename.split('_')[1].split('.')[0]  # e.g., '023'
    except IndexError:
        print(f"Filename format unexpected: {filename}")
        continue

    with open(filepath, 'r') as file:
        lines = file.readlines()
    
    for line in lines:
        line = line.strip()
        
        # Skip lines that don't start with data (e.g., path or RESET lines)
        if not line or not line[0].isdigit():
            continue
        
        parts = [p.strip() for p in line.split(',')]
        if len(parts) != 5:
            continue  # Skip malformed lines

        try:
            niskin = int(parts[0])
            bottle = int(parts[1])
            dt_str = parts[2]
            start_byte = int(parts[3])
            end_byte = int(parts[4])

            # Convert to datetime
            dt = datetime.strptime(dt_str, '%b %d %Y %H:%M:%S')
            
            all_bottle_data.append({
                'cruise': cruise_portion,
                'Cast': cast_portion,
                'Niskin': niskin,
                'bottle': bottle,
                'date': dt.date(),
                'time': dt.time(),
                'datetime': dt,
                'start_byte': start_byte,
                'end_byte': end_byte,
                'source_file': filename
            })
        except Exception as e:
            print(f"Error parsing line in {filename}: {line} — {e}")

# Create DataFrame
df = pd.DataFrame(all_bottle_data)

# Preview result
print(f"Parsed {len(df)} rows from {len(bl_files)} files.")
df.head()

output_path = f'./Bottle_file_copies/merged_bottle_data_csvs/{cruise}_merged_bottle_data.csv'
df.to_csv(output_path, index=False)

print(f"DataFrame saved to {output_path}")"""

# Or read in and look at the merged bottle file directly
#merged_bottle_file.columns.tolist()
merged_bottle_file.head()

,Cast,Bottle Position,Start Time [UTC],Start Longitude [degrees],Start Latitude [degrees],Filename,Date Time,"Pressure, Digiquartz [db]","Depth [salt water, m]",Latitude [deg],...,"Conductivity, 2 [S/m]","Salinity, Practical [PSU]","Salinity, Practical, 2 [PSU]","Oxygen raw, SBE 43 [V]","Oxygen, SBE 43 [ml/l]","Oxygen Saturation, Garcia & Gordon [ml/l]","Beam Attenuation, WET Labs C-Star [1/m]","Beam Transmission, WET Labs C-Star [%]","Fluorescence, WET Labs ECO-AFL/FL [mg/m^3]",Cruise
0,1,1,2025-11-01T13:16:10.000Z,-75.126,35.9495,ar98a_001.hex,2025-11-01T13:19:22.000Z,27.506,27.303,35.94956,...,4.422975,33.3659,33.3639,2.6526,5.0377,5.39850,0.3969,90.5543,0.7093 (avg),AR98A
1,1,2,2025-11-01T13:16:10.000Z,-75.126,35.9495,ar98a_001.hex,2025-11-01T13:19:52.000Z,27.519,27.316,35.94956,...,4.422903,33.3659,33.3642,2.6272,4.9617,5.39858,0.3963,90.5682,0.7562 (avg),AR98A
2,1,3,2025-11-01T13:16:10.000Z,-75.126,35.9495,ar98a_001.hex,2025-11-01T13:23:05.000Z,20.235,20.086,35.94958,...,4.422676,33.3651,33.3642,2.6433,5.0077,5.39845,0.3931,90.6391,0.7613 (avg),AR98A
3,1,4,2025-11-01T13:16:10.000Z,-75.126,35.9495,ar98a_001.hex,2025-11-01T13:23:30.000Z,20.544,20.393,35.94958,...,4.422556,33.3644,33.3642,2.6360,4.9850,5.39873,0.3981,90.5263,0.7490 (avg),AR98A
4,1,5,2025-11-01T13:16:10.000Z,-75.126,35.9495,ar98a_001.hex,2025-11-01T13:26:33.000Z,7.306,7.252,35.94957,...,4.422333,33.3648,33.3658,2.6688,5.0659,5.39861,0.3907,90.6942,0.7296 (avg),AR98A


In [26]:
processed_discrete_ifcb_data.head()

,Filename,HDR Comment,Sample Type,Cruise Leg,Site,Cast,Niskin,IFCB Bottle,Target Cast Depth,Trip Depth,# Triggers,# ROIs,Run Time,Inhibit Time,Sample Time,Volume Analyzed,ROIs/ml,Cast Start Latitude,Cast Start Longitude,Notes
0,D20251101T164030_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,1,6,2,7,7.3,3698,3381,1200.866944,306.844688,894.022257,3.725093,907.628410,35.949567,-75.126083,LTER IFCB
1,D20251101T170419_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,1,6,2,7,7.3,3631,3309,1200.964583,300.425556,900.539028,3.752246,881.871830,35.949567,-75.126083,LTER IFCB
2,D20251101T172810_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,1,6,2,7,7.3,3705,3396,1201.778889,306.580000,895.198889,3.729995,910.456894,35.949567,-75.126083,LTER IFCB
3,D20251101T180159_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,1,2,11,bottom,27.0,3419,3144,1200.889583,283.876562,917.013021,3.820888,822.845459,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample
4,D20251101T183013_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,1,2,11,bottom,27.0,3422,3090,1200.994167,284.636094,916.358073,3.818159,809.290628,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample


In [28]:
# Make sure both are strings and pad the processed_discrete_ifcb_data to match the 3-digit format in df
processed_discrete_ifcb_data['Cast'] = processed_discrete_ifcb_data['Cast'].astype(str).str.zfill(3)
processed_discrete_ifcb_data['Cruise Leg'] = processed_discrete_ifcb_data['Cruise Leg'].str.lower()

merged_bottle_file['Cast'] = merged_bottle_file['Cast'].astype(str).str.zfill(3)  # Just to be safe
# Parse out leg from cruise number
merged_bottle_file['Cruise Leg'] = merged_bottle_file['Cruise'].str.strip().str[-1]
merged_bottle_file['Cruise Leg'] = merged_bottle_file['Cruise Leg'].str.lower()

merged_bottle_file = merged_bottle_file.rename(columns={'Bottle Position': 'Niskin'})

# Now merge
merged = processed_discrete_ifcb_data.merge(
    merged_bottle_file[['Cruise Leg', 'Cast', 'Niskin', 'Date Time', 'Depth [salt water, m]']],
    on=['Cruise Leg', 'Cast', 'Niskin'],
    how='left'
)

# Rename cols for clarity
merged = merged.rename(columns={'Date Time': 'bottle_datetime'})
merged = merged.rename(columns={'Depth [salt water, m]': 'bottle_depth'})


# Preview
merged.head()
#merged_bottle_file.head()

,Filename,HDR Comment,Sample Type,Cruise Leg,Site,Cast,Niskin,IFCB Bottle,Target Cast Depth,Trip Depth,...,Run Time,Inhibit Time,Sample Time,Volume Analyzed,ROIs/ml,Cast Start Latitude,Cast Start Longitude,Notes,bottle_datetime,bottle_depth
0,D20251101T164030_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,001,6,2,7,7.3,...,1200.866944,306.844688,894.022257,3.725093,907.628410,35.949567,-75.126083,LTER IFCB,2025-11-01T13:27:01.000Z,7.416
1,D20251101T170419_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,001,6,2,7,7.3,...,1200.964583,300.425556,900.539028,3.752246,881.871830,35.949567,-75.126083,LTER IFCB,2025-11-01T13:27:01.000Z,7.416
2,D20251101T172810_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,001,6,2,7,7.3,...,1201.778889,306.580000,895.198889,3.729995,910.456894,35.949567,-75.126083,LTER IFCB,2025-11-01T13:27:01.000Z,7.416
3,D20251101T180159_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,001,2,11,bottom,27.0,...,1200.889583,283.876562,917.013021,3.820888,822.845459,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample,2025-11-01T13:19:52.000Z,27.316
4,D20251101T183013_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,001,2,11,bottom,27.0,...,1200.994167,284.636094,916.358073,3.818159,809.290628,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample,2025-11-01T13:19:52.000Z,27.316


In [29]:
processed_discrete_ifcb_data_merged = merged
processed_discrete_ifcb_data_merged

,Filename,HDR Comment,Sample Type,Cruise Leg,Site,Cast,Niskin,IFCB Bottle,Target Cast Depth,Trip Depth,...,Run Time,Inhibit Time,Sample Time,Volume Analyzed,ROIs/ml,Cast Start Latitude,Cast Start Longitude,Notes,bottle_datetime,bottle_depth
0,D20251101T164030_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,001,6,2,7,7.3,...,1200.866944,306.844688,894.022257,3.725093,907.628410,35.949567,-75.126083,LTER IFCB,2025-11-01T13:27:01.000Z,7.416
1,D20251101T170419_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,001,6,2,7,7.3,...,1200.964583,300.425556,900.539028,3.752246,881.871830,35.949567,-75.126083,LTER IFCB,2025-11-01T13:27:01.000Z,7.416
2,D20251101T172810_IFCB199,ar98a_c1n6_b2_cn_7m,discrete_water_sample,a,CP10CNSM,001,6,2,7,7.3,...,1201.778889,306.580000,895.198889,3.729995,910.456894,35.949567,-75.126083,LTER IFCB,2025-11-01T13:27:01.000Z,7.416
3,D20251101T180159_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,001,2,11,bottom,27.0,...,1200.889583,283.876562,917.013021,3.820888,822.845459,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample,2025-11-01T13:19:52.000Z,27.316
4,D20251101T183013_IFCB199,ar98a_c1n2_b11_cn_bottom,discrete_water_sample,a,CP10CNSM,001,2,11,bottom,27.0,...,1200.994167,284.636094,916.358073,3.818159,809.290628,35.949567,-75.126083,LTER IFCB; IFCB bottle 11 was LTER IFCB sample,2025-11-01T13:19:52.000Z,27.316
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,D20251116T014257_IFCB199,ar98b_c4n5_b2_cn_7m,discrete_water_sample,b,CP12CNSW,004,5,2,7,7.4,...,1200.884444,207.981337,992.903108,4.137096,599.212547,35.949933,-75.128200,,NaN,NaN
82,D20251116T020646_IFCB199,ar98b_c4n5_b2_cn_7m,discrete_water_sample,b,CP12CNSW,004,5,2,7,7.4,...,1200.947917,202.684010,998.263906,4.159433,578.925068,35.949933,-75.128200,,NaN,NaN
83,D20251116T023347_IFCB199,ar98b_c4n1_b1_cn_bottom,discrete_water_sample,b,CP12CNSW,004,1,1,bottom,27.3,...,1200.907917,166.444097,1034.463819,4.310266,457.512376,35.949933,-75.128200,,NaN,NaN
84,D20251116T025737_IFCB199,ar98b_c4n1_b1_cn_bottom,discrete_water_sample,b,CP12CNSW,004,1,1,bottom,27.3,...,1200.916111,164.649861,1036.266250,4.317776,447.684174,35.949933,-75.128200,,NaN,NaN


In [33]:
# handling samples associated with 2 differing sites for underway data

def parse_sites(site_val):
    if pd.isna(site_val) or site_val == '':
        return '', ''
    
    # split on semicolon and clean whitespace
    sites = [s.strip() for s in str(site_val).split(';') if s.strip() != '']
    
    # format as site_*
    sites = ['site_' + s for s in sites]
    
    # assign to tag1 and tag3
    tag1 = sites[0] if len(sites) >= 1 else ''
    tag3 = sites[1] if len(sites) >= 2 else ''
    
    return tag1, tag3

site_tags = processed_underway_ifcb_data['Site'].apply(parse_sites)

0      (, )
1      (, )
2      (, )
3      (, )
4      (, )
       ... 
984    (, )
985    (, )
986    (, )
987    (, )
988    (, )
Name: Site, Length: 989, dtype: object

In [60]:
# handling qc tags

def get_qc_tags(row):
    qc_tags = []

    sample_type = str(row.get('Sample Type', '')).lower() if pd.notna(row.get('Sample Type', '')) else ''
    hdr_comment = str(row.get('HDR Comment', '')).lower() if pd.notna(row.get('HDR Comment', '')) else ''
    notes = str(row.get('Notes', '')).lower() if pd.notna(row.get('Notes', '')) else ''

    # Existing QC logic
    if 'test' in sample_type or 'test' in hdr_comment:
        qc_tags.append('qc_test')
    if 'beads' in sample_type or 'beads' in hdr_comment:
        qc_tags.append('qc_beads')

    # Notes-based QC logic
    if 'focus' in notes:
        qc_tags.append('qc_bad_focus')
    # Handle debubble BEFORE bubble to avoid substring conflicts
    if 'debubble' in notes or 'debubbled' in notes:
        qc_tags.append('qc_ifcb_debubbled')
    elif 'bubble' in notes or 'bubbles' in notes:
        qc_tags.append('qc_bubbles_in_ROIs')
    if 'stopped' in notes or 'stopped_early' in notes:
        qc_tags.append('qc_stopped_early')
    if 'bad_data' in notes:
        qc_tags.append('qc_bad_data')

    # remove duplicates while preserving order
    qc_tags = list(dict.fromkeys(qc_tags))

    return qc_tags

In [61]:
columns_in_metadata_csv = ['filename', 'Latitude', 'Longitude', 'Depth', 'sample_type', 'Cruise', 'Instrument', 'datetime', 'tag1', 'tag2', 'tag3']

# Parse site tags for underway
site_tags_underway = processed_underway_ifcb_data['Site'].apply(parse_sites)

processed_underway_ifcb_data_mapped = {
    'filename': processed_underway_ifcb_data['Filename'], 
    'Latitude': processed_underway_ifcb_data['Ship Latitude'],
    'Longitude': processed_underway_ifcb_data['Ship Longitude'],
    'Depth': 2.13, #for R/V Armstrong Aft Diaphram Pump
    'sample_type': 'underway',
    'Cruise': cruise,
    'Instrument': ifcb_instrument_num,
    'datetime': '',
    #'tag1': processed_underway_ifcb_data['Site'].apply(lambda x: 'site_' + str(x) if pd.notna(x) and x != '' else x), 
    'tag1': site_tags.apply(lambda x: x[0]),
    'tag2': 'targetdepth_surface',
    'tag3': site_tags.apply(lambda x: x[1])
}

processed_discrete_ifcb_data_mapped = {
    'filename': processed_discrete_ifcb_data_merged['Filename'], 
    'Latitude': processed_discrete_ifcb_data_merged['Cast Start Latitude'],
    'Longitude': processed_discrete_ifcb_data_merged['Cast Start Longitude'],
    'Depth': processed_discrete_ifcb_data_merged['bottle_depth'],
    'sample_type': processed_discrete_ifcb_data_merged['Sample Type'].apply(
        lambda x: 'discrete' if 'discrete' in str(x).lower() and 'test' not in str(x).lower() and 'beads' not in str(x).lower() else x
    ),
    'Cruise': cruise,
    'Instrument':ifcb_instrument_num,
    'datetime': processed_discrete_ifcb_data_merged['bottle_datetime'],
    'tag1': processed_discrete_ifcb_data_merged['Site'].apply(lambda x: 'site_' + str(x) if pd.notna(x) and x != '' else x), 
    'tag2': processed_discrete_ifcb_data_merged['Target Cast Depth'].apply(lambda x: 'targetdepth_' + str(x) if pd.notna(x) and x != '' else x), 
    'tag3': ''
}

# Generate QC tags lists
qc_tag_lists_underway = processed_underway_ifcb_data.apply(get_qc_tags, axis=1)
qc_tag_lists_discrete = processed_discrete_ifcb_data_merged.apply(get_qc_tags, axis=1)

# Find max QC tag counts
max_qc_tags_underway = qc_tag_lists_underway.apply(len).max()
max_qc_tags_discrete = qc_tag_lists_discrete.apply(len).max()

# Add tag4, tag5, ... as needed for underway
for i in range(max_qc_tags_underway):
    processed_underway_ifcb_data_mapped[f'tag{i+4}'] = qc_tag_lists_underway.apply(
        lambda tags: tags[i] if i < len(tags) else ''
    )

# Add tag4, tag5, ... as needed for discrete
for i in range(max_qc_tags_discrete):
    processed_discrete_ifcb_data_mapped[f'tag{i+4}'] = qc_tag_lists_discrete.apply(
        lambda tags: tags[i] if i < len(tags) else ''
    )

# Convert to DataFrames
underway_new = pd.DataFrame(processed_underway_ifcb_data_mapped).fillna('')
discrete_new = pd.DataFrame(processed_discrete_ifcb_data_mapped).fillna('')

# Combine
metadata_df = pd.concat([underway_new, discrete_new], ignore_index=True).fillna('')

pd.set_option('display.max_rows', None)
metadata_df

,filename,Latitude,Longitude,Depth,sample_type,Cruise,Instrument,datetime,tag1,tag2,tag3,tag4,tag5,tag6
0,D20251030T170909_IFCB199,41.210000,-71.233000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
1,D20251030T173245_IFCB199,41.158000,-71.298000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
2,D20251030T175621_IFCB199,41.104000,-71.367000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
3,D20251030T181956_IFCB199,41.050000,-71.436000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
4,D20251030T184331_IFCB199,40.999000,-71.502000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
5,D20251030T190706_IFCB199,40.951000,-71.577000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
6,D20251030T193041_IFCB199,40.908000,-71.654000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
7,D20251030T195417_IFCB199,40.864000,-71.734000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
8,D20251030T201752_IFCB199,40.818000,-71.812000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,
9,D20251030T204127_IFCB199,40.775000,-71.888000,2.13,underway,AR98,IFCB199,,,targetdepth_surface,,,,


In [62]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M%S")
output_filename = f"Metadata_CSVs/{cruise}_shipboard_ifcb_dashboard_metadata_{timestamp}.csv"
metadata_df.to_csv(output_filename, index=False)